# Training the detector (Google Colab)

Trains a YOLO detector for the two target classes used by this project
(`mannequin`, `tent`) and produces the weights file the flight code loads (`MODEL_FILE`).

**Before you start**

1. `Runtime > Change runtime type > GPU` (an A100 is ideal for `imgsz=1280`).
2. Have a dataset in YOLO format with a `data.yaml`. Any source works — Roboflow,
   CVAT, or your own folders — as long as the class names match `CLASS_NAMES` in
   `suas/config.py`.

Nothing here is specific to this repository's dataset: point the paths below at
your own data.

## 1) Install and check the GPU

In [ ]:
%pip install -q ultralytics
import torch, ultralytics
ultralytics.checks()
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - select a GPU runtime')

## 2) Settings

Edit these three lines and run everything below unchanged.

In [ ]:
# Where checkpoints and the final weights are written.
# Google Drive survives a disconnected Colab session; /content does not.
PROJECT_DIR = '/content/drive/MyDrive/uav_detector'

# Your dataset's data.yaml (filled in by the next cell if you download one).
DATA = '/content/dataset/data.yaml'

RUN_NAME = 'train1'
IMGSZ    = 1280        # large, because targets are small from altitude
EPOCHS   = 100

from google.colab import drive
drive.mount('/content/drive')

## 3) Get the dataset

Replace this cell with whatever your dataset needs. Two common cases:

```python
# A) Roboflow export (paste YOUR OWN download link - it contains a private key)
!curl -L -sS 'PASTE_YOUR_ROBOFLOW_DOWNLOAD_URL_HERE' -o /content/data.zip
!mkdir -p /content/dataset && unzip -q -o /content/data.zip -d /content/dataset

# B) A zip already in your Drive
!unzip -q -o '/content/drive/MyDrive/my_dataset.zip' -d /content/dataset
```

> Never commit a dataset download link to a public repository: it embeds an API key.

In [ ]:
import glob, os
assert os.path.exists(DATA), f'data.yaml not found at {DATA} - download your dataset first'
print(open(DATA).read())
print('images:', len(glob.glob('/content/dataset/**/images/*.*', recursive=True)))

## 4) Train

Augmentation is on: aerial targets appear at every rotation and scale, so
`degrees`, `flipud` and `scale` matter more here than in a typical dataset.
`save_period` writes a checkpoint every 5 epochs, so a dropped session is
recoverable.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11m.pt')          # yolo11n/s are faster; m is a good accuracy trade-off
results = model.train(
    data=DATA, epochs=EPOCHS, patience=25, imgsz=IMGSZ, batch=-1,
    scale=0.5, fliplr=0.5, flipud=0.5, degrees=15.0, translate=0.1,
    mosaic=1.0, close_mosaic=10, mixup=0.1, copy_paste=0.0, erasing=0.0,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, multi_scale=False,
    amp=True, cache='disk', workers=8, cos_lr=True, plots=True,
    save=True, save_period=5,
    project=f'{PROJECT_DIR}/runs', name=RUN_NAME, exist_ok=True,
)
BEST = f'{PROJECT_DIR}/runs/{RUN_NAME}/weights/best.pt'
print('best.pt ->', BEST)

### Resume after a disconnect

In [ ]:
# from ultralytics import YOLO
# YOLO(f'{PROJECT_DIR}/runs/{RUN_NAME}/weights/last.pt').train(resume=True)

## 5) Validate

Watch the **per-class recall**. A class the model rarely finds is worse than one it
occasionally gets wrong: the mission only acts on what it detects. If one class is a
small minority in your data, oversample it and retrain.

In [ ]:
metrics = model.val(data=DATA, imgsz=IMGSZ)
print('mAP50:', round(float(metrics.box.map50), 4),
      '| mAP50-95:', round(float(metrics.box.map), 4))
for i, name in metrics.names.items():
    print(f'  {name:12s} P={metrics.box.p[i]:.3f} '
          f'R={metrics.box.r[i]:.3f} mAP50={metrics.box.ap50[i]:.3f}')

## 6) Sanity check on a real aerial image

Model metrics on a held-out split can still hide a domain gap. Try a photo taken by
the actual aircraft camera, at the altitude you plan to fly.

In [ ]:
from google.colab import files
from google.colab.patches import cv2_imshow

up  = files.upload()
img = list(up.keys())[0]

r = model.predict(img, imgsz=IMGSZ, conf=0.25)
cv2_imshow(r[0].plot())
print('detections:', [model.names[int(c)] for c in r[0].boxes.cls],
      '| conf:', [round(float(x), 2) for x in r[0].boxes.conf])

## 7) Deploy to the companion computer

1. Download `best.pt` and copy it to `models/` on the Jetson and set `MODEL_FILE` in `suas/config.py`.
2. Make sure `CLASS_NAMES` in `suas/config.py` matches the model's class names — the
   detector logs a loud warning on a mismatch, because renamed or reordered classes
   silently break target selection and the payload identity gate.
3. Optionally build a TensorRT engine **on the Jetson itself** (never in Colab, and
   never copied between machines):

```bash
python3 tools/export_tensorrt.py
```

4. Verify end to end before flying:

```bash
python3 -m unittest discover -s tests
python3 tools/preview.py
```